## Prerequisites

Runtime: Python 3, T4 GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Florence-2 requires timm (vision backbone) and einops.
# Pin tokenizers<0.21: Florence-2's custom TokenizersBackend uses an API
# removed in tokenizers 0.21+ ("additional_special_tokens" attribute error).
%pip install -q "transformers>=4.41.0" "tokenizers<0.21" timm einops Pillow

In [3]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [4]:
import torch

In [5]:
from pathlib import Path
from PIL import Image

WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')
MODEL_NAME = 'microsoft/Florence-2-large'

# Florence-2-large: 0.77B params → ~1.5 GB fp16, well within T4's 15 GB.
# No quantization needed.
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).eval().cuda()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

configuration_florence2.py: 0.00B [00:00, ?B/s]

You are using a model of type florence2 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

AttributeError: TokenizersBackend has no attribute additional_special_tokens

In [ ]:
IMAGE_FILE = WORKING_DIR / 'images/pineda1/pineda1_page_3.png'

## Inference

In [ ]:
import time

image_stem   = IMAGE_FILE.stem
image_folder = IMAGE_FILE.parent.name

image = Image.open(IMAGE_FILE).convert('RGB')

# Florence-2 uses task tokens rather than free-form prompts.
# <OCR> returns the full text content of the image as plain text.
task_prompt = '<OCR>'
inputs = processor(
    text=task_prompt, images=image, return_tensors='pt'
).to('cuda', torch.float16)

t0 = time.time()
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs['input_ids'],
        pixel_values=inputs['pixel_values'],
        max_new_tokens=4096,
        do_sample=False,
        num_beams=3,
    )
elapsed = time.time() - t0

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
parsed = processor.post_process_generation(
    generated_text,
    task=task_prompt,
    image_size=(image.width, image.height),
)
transcription = parsed[task_prompt]

print(f'Done in {elapsed:.1f}s')
print(transcription)

### Saving the output

In [ ]:
# Transcription → transcriptions/Florence-2-large/<stem>.md
transcription_out = WORKING_DIR / 'transcriptions/Florence-2-large'
transcription_out.mkdir(parents=True, exist_ok=True)
(transcription_out / f'{image_stem}.md').write_text(transcription, encoding='utf-8')
print(f'Saved: transcriptions/Florence-2-large/{image_stem}.md')